# CEFOMENA — entrenamiento en Colab (solo español)

1. Entorno → GPU (T4/L4/A100). 2. Opcional: añade el secreto `HF_TOKEN` (🔑 en la barra lateral) para publicar.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: el entrenamiento será lento"
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!git clone --depth 1 https://github.com/Jessbludev/Neural-CEFOMENA.git
%cd Neural-CEFOMENA
# Usa el torch con CUDA que ya trae Colab (--no-sources evita el índice CPU de pyproject).
!uv pip install --system --no-sources -e ".[hub]"
!python -c "import torch, cefomena; print(torch.__version__, torch.cuda.is_available(), cefomena.__version__)"

## Parámetros

In [ ]:
PRESET = "tiny"  # tiny ≈30,8 M | micro ≈9,3 M
STEPS = 20000
BATCH = 32
CONTEXT = 512
VOCAB = 8192
SYNTH_N = 20000
HF_REPO = "Jessbludev/cefomena-tiny"

## Datos en español

In [ ]:
!mkdir -p data runs
!cefomena synth --n {SYNTH_N} --out data/synth.jsonl
!cefomena oasst-es --out data/oasst_es.jsonl
!wc -l data/*.jsonl

## Tokenizador y dataset

In [ ]:
!cefomena train-tokenizer --data data/synth.jsonl data/oasst_es.jsonl --vocab-size {VOCAB} --out data/tok.json
!cefomena prepare --data data/synth.jsonl data/oasst_es.jsonl --tokenizer data/tok.json --out data/train.npz

## Entrenamiento (MetabolicFSM)

In [ ]:
!cefomena train --data data/train.npz --tokenizer data/tok.json --preset {PRESET} \
    --steps {STEPS} --batch-size {BATCH} --context {CONTEXT} --device auto --out runs/{PRESET}

## Prueba rápida: chat y agente (dryrun)

In [ ]:
from cefomena.chat.generate import ChatSession
from cefomena.export.safetensors_io import load_pretrained

model, tok = load_pretrained(f"runs/{PRESET}")
session = ChatSession(model, tok)
for q in ["Hola, ¿qué tal?", "¿Cuánto es 12 * 7?", "Ayúdame a organizar mi día"]:
    r = session.reply(q)
    print("tú>", q)
    print("cefomena>", r.content or r.tool_call)

In [ ]:
!cefomena agent --model runs/{PRESET} --task "Escribe «hola mundo» en la ventana abierta" --backend dryrun

## Integridad y publicación en Hugging Face

In [ ]:
!cd runs/{PRESET} && sha256sum -c SHA256SUMS
!cp docs/MODEL_CARD.md runs/{PRESET}/README.md
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!cefomena push --model runs/{PRESET} --repo-id {HF_REPO}